In [3]:
import glob
import pandas as pd
from upsetplot import UpSet, from_indicators
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
# Alle CSV-Dateien, die recording_*.csv heißen
files = glob.glob("./recordings/recording_*.csv")

print("Gefundene Dateien:")
for f in files:
    print("  →", f)

data = pd.DataFrame()

for f in files:
    df = pd.read_csv(f)

    # Dictionary für eine Zeile (eine Datei)
    row = {"filename": f}

    # Jede Spalte prüfen
    for col in df.columns:
        # Versuchen, die Spalte numerisch umzuwandeln
        numeric_col = pd.to_numeric(df[col], errors="coerce")

        # Nur verwenden, wenn mindestens ein Wert wirklich numerisch ist
        if numeric_col.notna().any():
            row[f"{col}_mean"] = numeric_col.mean()
            row[f"{col}_min"]  = numeric_col.min()
            row[f"{col}_max"]  = numeric_col.max()

    # Zeile ins Ergebnis einfügen
    data = pd.concat([data, pd.DataFrame([row])], ignore_index=True)

print("\nFertiger DataFrame:")
print(data)



Gefundene Dateien:
  → ./recordings\recording_2025_12_11__12_29_26_florian.csv
  → ./recordings\recording_2025_12_11__12_38_11_matthias.csv
  → ./recordings\recording_2025_12_11__12_46_32_fabian.csv
  → ./recordings\recording_2026_02_10__13_10_22_fabian.csv
  → ./recordings\recording_2026_02_10__13_18_02_florian.csv
  → ./recordings\recording_2026_02_10__13_25_22_matthias.csv
  → ./recordings\recording_2026_02_10__13_37_56_florian.csv
  → ./recordings\recording_2026_02_10__13_44_54_matthias.csv
  → ./recordings\recording_2026_02_10__13_51_14_fabian.csv
  → ./recordings\recording_2026_02_10__14_03_00_florian_night.csv
  → ./recordings\recording_2026_02_10__14_10_05_matthias_night.csv
  → ./recordings\recording_2026_02_10__14_16_19_fabian_night.csv
  → ./recordings\recording_2026_02_10__14_22_43_florian_night.csv
  → ./recordings\recording_2026_02_10__14_29_48_matthias_night.csv
  → ./recordings\recording_2026_02_10__15_13_03_florian.csv
  → ./recordings\recording_2026_02_10__15_19_00_ma

C:\Users\games\AppData\Local\Temp\ipykernel_31280\1557897203.py:11: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,15,16,17,18,19,24,27,31,35,39,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
C:\Users\games\AppData\Local\Temp\ipykernel_31280\1557897203.py:11: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,20,23,27,31,35,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,71,72,73,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
C:\Users\games\AppData\Local\Temp\ipykernel_31280\1557897203.py:11: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,18,21,25,29,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,69,70,71,72,73,78) have mixed types. Specify dtype option on import or set low_memory=False.
  


Fertiger DataFrame:
                                             filename  wheel_position_mean  \
0   ./recordings\recording_2025_12_11__12_29_26_fl...             0.165776   
1   ./recordings\recording_2025_12_11__12_38_11_ma...             0.303292   
2   ./recordings\recording_2025_12_11__12_46_32_fa...             0.218411   
3   ./recordings\recording_2026_02_10__13_10_22_fa...             0.289260   
4   ./recordings\recording_2026_02_10__13_18_02_fl...            -0.733170   
5   ./recordings\recording_2026_02_10__13_25_22_ma...             0.338776   
6   ./recordings\recording_2026_02_10__13_37_56_fl...            -0.013013   
7   ./recordings\recording_2026_02_10__13_44_54_ma...            -0.778495   
8   ./recordings\recording_2026_02_10__13_51_14_fa...             0.217672   
9   ./recordings\recording_2026_02_10__14_03_00_fl...            -0.337546   
10  ./recordings\recording_2026_02_10__14_10_05_ma...             1.161311   
11  ./recordings\recording_2026_02_10__14_1

In [5]:
numeric_cols = data.select_dtypes(include="number").columns

for col in numeric_cols:
    fig = px.box(
        data,
        y=col,
        points="all",
        hover_data=["filename"],
        title=f"Boxplot mit Ausreißern: {col}"
    )

    fig.update_traces(jitter=0.35, boxmean=True)

    # Speichern als HTML
    fig.write_html(f"./plots/all-town-recordings/joint/boxplot_{col}.html")

In [6]:
long_df = data.melt(
    id_vars="filename",
    value_vars=numeric_cols,
    var_name="metric",
    value_name="value"
)

fig = px.box(
    long_df,
    x="metric",
    y="value",
    points="all",
    hover_data=["filename"],
    title="Alle Spalten als Boxplots (interaktiv)"
)

fig.update_traces(jitter=0.3)
fig.show()

In [7]:
fig.write_html("./plots/all-town-recordings/joint/all_boxplots.html")